            # Bulk RNA-seqで免疫の状態を読む: SLEのIFN signature

            この実習では、免疫細胞ごとのbulk RNA-seqカウントデータを使って、HCとSLEを比較します。

            今日のゴールはRの文法を覚えることではなく、図と遺伝子名から「どの免疫応答が動いているのか」を読むことです。

            - 1回目: single-cellデータでmarker発現から細胞集団に名前をつけた
            - 2回目: batch effect correctionで、データのずれを補正する考え方を見た
            - 3回目: bulk RNA-seqで、疾患群と健常群の違いを遺伝子発現から読む
            

            ## セル1: 環境セットアップ

            そのまま実行してください。  
            通常はGoogle Driveから `R_lib.zip` と実習データをColabの `/content` に自動ダウンロードして解凍します。

            自動ダウンロードに失敗した場合だけ、左側のファイル欄から次の2ファイルを `/content` にアップロードして、このセルを再実行してください。

            - `R_lib.zip`
            - `bulk_deg_course_data.zip`
            

In [ ]:
            options(bitmapType = "cairo")
            set.seed(123)

            # ===== 講師設定欄 =====
            # Google Driveで「リンクを知っている全員が閲覧可」にしたフォルダURLです。
            # フォルダ内に R_lib.zip と bulk_deg_course_data.zip を置いてください。
            # 学生はここを編集しません。
            DRIVE_FOLDER_URL <- "https://drive.google.com/drive/folders/1yoL636iOkm1N9SALpJa6GyjVZzkwzjzp?usp=sharing"

            # 個別ファイルIDで配布したい場合だけ使います。通常は上のフォルダURLだけでOKです。
            R_LIB_FILE_ID <- "REPLACE_WITH_R_LIB_ZIP_FILE_ID"
            DATA_FILE_ID  <- "REPLACE_WITH_DATA_ZIP_FILE_ID"

            # SHA256は任意です。設定しておくとファイル破損を検出できます。
            R_LIB_SHA256 <- "edd15eb4d459d7c1d042be7057eb1b8d5c5d4e74d4ad87a6b8171a447be41546"
            DATA_SHA256  <- "d28ec8b6faa74612652f03b9a7d79a01653baef33c788625d75cbaeffbd00edc"

            R_LIB_ZIP <- "/content/R_lib.zip"
            DATA_ZIP <- "/content/bulk_deg_course_data.zip"
            COURSE_DIR <- "/content/bulk_deg_course_data"
            DOWNLOAD_DIR <- "/content/course_drive_files"

            drive_download <- function(file_id, dest) {
              if (!nzchar(file_id) || grepl("^REPLACE_WITH_", file_id)) {
                return(FALSE)
              }
              cat("Google Driveから取得します:", basename(dest), "\n")
              system2("python3", c("-m", "pip", "install", "-q", "gdown"))
              url <- paste0("https://drive.google.com/uc?id=", file_id)
              status <- system2("python3", c("-m", "gdown", url, "-O", dest))
              identical(status, 0L) || identical(status, 0)
            }

            folder_download <- function(folder_url) {
              if (!nzchar(folder_url) || grepl("^REPLACE_WITH_", folder_url)) {
                return(FALSE)
              }
              cat("Google Driveフォルダから実習ファイルを取得します。\n")
              system2("python3", c("-m", "pip", "install", "-q", "gdown"))
              dir.create(DOWNLOAD_DIR, showWarnings = FALSE, recursive = TRUE)
              status <- system2("python3", c("-m", "gdown", "--folder", folder_url, "-O", DOWNLOAD_DIR, "--remaining-ok"))
              identical(status, 0L) || identical(status, 0)
            }

            find_downloaded_file <- function(filename) {
              if (!dir.exists(DOWNLOAD_DIR)) return(NA_character_)
              hits <- list.files(DOWNLOAD_DIR, pattern = paste0("^", filename, "$"), recursive = TRUE, full.names = TRUE)
              if (length(hits) == 0) NA_character_ else hits[1]
            }

            folder_downloaded <- FALSE

            ensure_file <- function(path, file_id, label) {
              if (!file.exists(path)) {
                ok <- drive_download(file_id, path)
                if (!ok || !file.exists(path)) {
                  if (!folder_downloaded) {
                    folder_downloaded <<- folder_download(DRIVE_FOLDER_URL)
                  }
                  downloaded <- find_downloaded_file(basename(path))
                  if (!is.na(downloaded)) {
                    file.copy(downloaded, path, overwrite = TRUE)
                  }
                }
                if (!file.exists(path)) {
                  stop(
                    label, " が見つかりません。\n",
                    "自動ダウンロードに失敗した可能性があります。\n",
                    "左側のファイル欄から ", basename(path), " を /content にアップロードして、このセルを再実行してください。"
                  )
                }
              }
              cat(label, ":", round(file.size(path) / 1e6, 1), "MB\n")
            }

            check_sha256 <- function(path, expected) {
              if (!nzchar(expected)) return(invisible(TRUE))
              actual <- unname(tools::sha256sum(path))
              if (!identical(tolower(actual), tolower(expected))) {
                stop(
                  "ファイルのSHA256が一致しません: ", basename(path), "\n",
                  "expected: ", expected, "\n",
                  "actual:   ", actual, "\n",
                  "ファイルが壊れている可能性があります。削除して再ダウンロードしてください。"
                )
              }
              cat("SHA256 OK:", basename(path), "\n")
            }

            ensure_file(R_LIB_ZIP, R_LIB_FILE_ID, "R library archive")
            ensure_file(DATA_ZIP, DATA_FILE_ID, "Course data")
            check_sha256(R_LIB_ZIP, R_LIB_SHA256)
            check_sha256(DATA_ZIP, DATA_SHA256)

            # R_lib.zipの解凍。zipの中身が R_lib または R_libs のどちらでも拾います。
            if (!dir.exists("/content/R_lib") && !dir.exists("/content/R_libs")) {
              cat("R_lib.zipを解凍中...\n")
              unzip(R_LIB_ZIP, exdir = "/content")
            }
            lib_candidates <- c("/content/R_lib", "/content/R_libs")
            lib_dir <- lib_candidates[dir.exists(lib_candidates)][1]
            if (is.na(lib_dir)) {
              stop("R_lib.zipを解凍しましたが、/content/R_lib または /content/R_libs が見つかりません。")
            }
            .libPaths(c(lib_dir, .libPaths()))
            cat("R library path:", .libPaths()[1], "\n")

            if (!dir.exists(COURSE_DIR)) {
              cat("実習データを解凍中...\n")
              unzip(DATA_ZIP, exdir = "/content")
            }
            if (!dir.exists(COURSE_DIR)) stop("実習データの解凍先が見つかりません: ", COURSE_DIR)

            suppressPackageStartupMessages({
              library(edgeR)
              library(ggplot2)
              library(ggrepel)
              library(pheatmap)
            })

            cat("\nセットアップ完了。edgeR version:", as.character(packageVersion("edgeR")), "\n")
            

            ## セル2: データの読み込み

            ここでは、解析に使うサンプル情報、細胞種リスト、IFN関連遺伝子リストを読み込みます。
            

In [ ]:
            meta <- read.table(file.path(COURSE_DIR, "sample_meta.tsv"), header = TRUE, sep = "	", stringsAsFactors = FALSE)
            celltype_info <- read.table(file.path(COURSE_DIR, "celltype_info.tsv"), header = TRUE, sep = "	", stringsAsFactors = FALSE)
            IFNgenes <- read.table(file.path(COURSE_DIR, "IFNgenes100.txt"), header = FALSE, stringsAsFactors = FALSE)[, 1]

            cat("サンプル数\n")
            print(table(meta$disease))
            cat("\n選べる細胞種\n")
            print(celltype_info[, c("celltype", "lineage")], row.names = FALSE)
            

            ## セル3: 解析する細胞種を選ぶ

            single-cellでは「細胞の集団」を見ました。  
            bulk RNA-seqでは、あらかじめ分けられた細胞種ごとに、HCとSLEの発現差を見ます。

            最初は `pDC` のまま実行してください。余裕があれば `CL_Mono`, `Th1`, `Plasmablast` などに変えて比較します。
            

In [ ]:
            celltype <- "pDC" #@param ["pDC", "CL_Mono", "mDC", "Th1", "Th17", "Plasmablast", "NK"]

            count_file <- file.path(COURSE_DIR, "count", paste0(celltype, "_count.tsv"))
            counts_raw <- read.table(count_file, header = TRUE, sep = "	", stringsAsFactors = FALSE, check.names = FALSE)

            gene_info <- counts_raw[, c("Gene_id", "Gene_name")]
            count_matrix <- counts_raw[, meta$sample_id]
            rownames(count_matrix) <- gene_info$Gene_id
            count_matrix <- as.matrix(count_matrix)

            cat("選んだ細胞種:", celltype, "\n")
            cat("遺伝子数:", nrow(count_matrix), "\n")
            cat("サンプル数:", ncol(count_matrix), "\n")
            

            ## セル4: DEG解析を実行

            そのまま実行してください。  
            ここでは `edgeR` を使って、SLEでHCより発現が高い/低い遺伝子を調べます。
            

In [ ]:
            group <- factor(meta$disease, levels = c("HC", "SLE"))

            dge <- DGEList(counts = count_matrix, group = group)
            keep <- filterByExpr(dge, group = group)
            dge <- dge[keep, , keep.lib.sizes = FALSE]
            dge <- calcNormFactors(dge, method = "TMM")

            design <- model.matrix(~ group)
            dge <- estimateDisp(dge, design)
            fit <- glmQLFit(dge, design)
            qlf <- glmQLFTest(fit, coef = "groupSLE")

            deg <- topTags(qlf, n = Inf)$table
            deg$Gene_id <- rownames(deg)
            deg$Gene_name <- gene_info$Gene_name[match(deg$Gene_id, gene_info$Gene_id)]
            deg <- deg[, c("Gene_id", "Gene_name", "logFC", "logCPM", "F", "PValue", "FDR")]

            log_cpm <- cpm(dge, log = TRUE, prior.count = 1)

            up_n <- sum(deg$FDR < 0.05 & deg$logFC > 0)
            down_n <- sum(deg$FDR < 0.05 & deg$logFC < 0)
            cat("SLEで発現上昇した遺伝子数 FDR<0.05:", up_n, "\n")
            cat("SLEで発現低下した遺伝子数 FDR<0.05:", down_n, "\n")
            

            ## セル5: PCAでサンプル全体の見え方を確認

            点が1人分のサンプルです。HCとSLEが完全に分かれる必要はありません。  
            「細胞種によって、疾患差が見えやすい/見えにくい」があることを見ます。
            

In [ ]:
            gene_var <- apply(log_cpm, 1, var)
            top_idx <- order(gene_var, decreasing = TRUE)[1:min(3000, length(gene_var))]
            pca <- prcomp(t(log_cpm[top_idx, ]), center = TRUE, scale. = TRUE)
            var_explained <- pca$sdev^2 / sum(pca$sdev^2)

            pca_df <- data.frame(
              sample_id = rownames(pca$x),
              PC1 = pca$x[, 1],
              PC2 = pca$x[, 2],
              disease = meta$disease[match(rownames(pca$x), meta$sample_id)]
            )

            options(repr.plot.width = 6.5, repr.plot.height = 5)
            ggplot(pca_df, aes(PC1, PC2, color = disease)) +
              geom_point(size = 3, alpha = 0.85) +
              scale_color_manual(values = c(HC = "#2f7d32", SLE = "#c7363d")) +
              labs(
                title = paste0(celltype, ": PCA"),
                x = paste0("PC1 (", round(var_explained[1] * 100, 1), "%)"),
                y = paste0("PC2 (", round(var_explained[2] * 100, 1), "%)")
              ) +
              theme_bw(base_size = 13)
            

            ## セル6: 上位DEGを見る

            `logFC` が正ならSLEで高い、負ならHCで高い、という意味です。
            

In [ ]:
            head(deg[order(deg$FDR), ], 15)
            

            ## セル7: Volcano plot

            右側にある点はSLEで高い遺伝子、左側にある点はHCで高い遺伝子です。  
            上にあるほど統計的に強い差があります。
            

In [ ]:
            volcano <- deg
            volcano$category <- "not significant"
            volcano$category[volcano$FDR < 0.05 & volcano$logFC > 0] <- "higher in SLE"
            volcano$category[volcano$FDR < 0.05 & volcano$logFC < 0] <- "higher in HC"
            volcano$minus_log10_fdr <- -log10(pmax(volcano$FDR, 1e-300))
            label_genes <- head(volcano[order(volcano$FDR), ], 12)

            options(repr.plot.width = 8, repr.plot.height = 5.5)
            ggplot(volcano, aes(logFC, minus_log10_fdr, color = category)) +
              geom_point(alpha = 0.55, size = 1.4) +
              geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "gray45") +
              geom_vline(xintercept = 0, color = "gray70") +
              geom_text_repel(data = label_genes, aes(label = Gene_name), size = 3, max.overlaps = 30) +
              scale_color_manual(values = c("higher in SLE" = "#c7363d", "higher in HC" = "#2764ad", "not significant" = "gray70")) +
              coord_cartesian(xlim = c(-6, 6)) +
              labs(
                title = paste0(celltype, ": SLE vs HC"),
                x = "log2 fold change: SLE / HC",
                y = "-log10 FDR",
                color = NULL
              ) +
              theme_bw(base_size = 13)
            

            ## セル8: IFN signatureを確認

            SLEでは、I型インターフェロン応答が上がることがよく知られています。  
            ここではIFN関連遺伝子が、SLEで上がったDEGの中に多いかを確認します。
            

In [ ]:
            expressed_gene_names <- gene_info$Gene_name[match(rownames(dge), gene_info$Gene_id)]
            expressed_gene_names <- unique(expressed_gene_names[!is.na(expressed_gene_names) & expressed_gene_names != ""])
            ifn_in_data <- intersect(IFNgenes, expressed_gene_names)

            up_genes <- unique(deg$Gene_name[deg$FDR < 0.05 & deg$logFC > 0])
            up_genes <- up_genes[!is.na(up_genes) & up_genes != ""]
            ifn_up <- intersect(up_genes, ifn_in_data)

            contingency <- matrix(c(
              length(ifn_up),
              length(up_genes) - length(ifn_up),
              length(ifn_in_data) - length(ifn_up),
              length(expressed_gene_names) - length(up_genes) - length(ifn_in_data) + length(ifn_up)
            ), nrow = 2, byrow = TRUE)
            rownames(contingency) <- c("IFN genes", "other genes")
            colnames(contingency) <- c("SLE-up DEG", "not SLE-up DEG")

            cat("発現データ中のIFN関連遺伝子数:", length(ifn_in_data), "\n")
            cat("SLEで上昇したIFN関連遺伝子数:", length(ifn_up), "\n\n")
            print(contingency)
            fisher_result <- fisher.test(contingency, alternative = "greater")
            cat("\nFisher検定 p値:", signif(fisher_result$p.value, 3), "\n")
            cat("オッズ比:", signif(unname(fisher_result$estimate), 3), "\n")
            

            ## セル9: IFN関連遺伝子の発現をヒートマップで見る

            赤いほど、そのサンプルで相対的に発現が高いことを示します。
            

In [ ]:
            top_ifn <- deg[deg$Gene_name %in% ifn_in_data, ]
            top_ifn <- head(top_ifn[order(top_ifn$FDR), ], 25)
            ifn_mat <- log_cpm[top_ifn$Gene_id, ]
            rownames(ifn_mat) <- top_ifn$Gene_name
            sample_order <- meta$sample_id[order(meta$disease)]
            ann <- data.frame(disease = meta$disease[match(sample_order, meta$sample_id)])
            rownames(ann) <- sample_order

            options(repr.plot.width = 7, repr.plot.height = 8)
            pheatmap(
              ifn_mat[, sample_order],
              scale = "row",
              cluster_cols = FALSE,
              annotation_col = ann,
              show_colnames = FALSE,
              main = paste0(celltype, ": IFN-related genes")
            )
            

            ## セル10: 気になる遺伝子を箱ひげ図で見る

            single-cell実習のFeaturePlotと同じ感覚で、特定の遺伝子の発現を確認します。
            

In [ ]:
            genes_to_check <- c("IFI27", "ISG15", "MX1", "OAS1", "CXCL10", "STAT1") #@param {type:"raw"}

            gene_rows <- gene_info$Gene_id[match(genes_to_check, gene_info$Gene_name)]
            ok <- !is.na(gene_rows) & gene_rows %in% rownames(log_cpm)
            plot_genes <- genes_to_check[ok]
            gene_rows <- gene_rows[ok]

            if (length(plot_genes) == 0) stop("指定した遺伝子がこのデータに見つかりません。")

            expr_df <- data.frame(
              sample_id = rep(colnames(log_cpm), each = length(plot_genes)),
              Gene = rep(plot_genes, times = ncol(log_cpm)),
              logCPM = as.vector(log_cpm[gene_rows, ])
            )
            expr_df$disease <- meta$disease[match(expr_df$sample_id, meta$sample_id)]

            options(repr.plot.width = 9, repr.plot.height = 4.8)
            ggplot(expr_df, aes(disease, logCPM, fill = disease)) +
              geom_boxplot(outlier.shape = NA, alpha = 0.7) +
              geom_jitter(width = 0.15, size = 1.4, alpha = 0.75) +
              facet_wrap(~ Gene, scales = "free_y", nrow = 1) +
              scale_fill_manual(values = c(HC = "#2f7d32", SLE = "#c7363d")) +
              labs(title = paste0(celltype, ": selected genes"), x = NULL, y = "log2 CPM") +
              theme_bw(base_size = 13) +
              theme(legend.position = "none")
            

            ## ミニ考察

            出席提出前に、下の3点を自分の中で確認してください。

            1. SLEで上がっている遺伝子には、どんな名前の遺伝子が多いですか。
            2. IFN関連遺伝子は、SLEで上がったDEGに多そうですか。
            3. その変化は、どの免疫細胞で特に見えやすそうですか。
            

            ## 最後のセル: 氏名と実行時刻

            氏名を漢字で入力してから実行してください。  
            実行後、このノートブックを `.ipynb` としてダウンロードし、指定された学生用サイトにアップロードしてください。
            

In [ ]:
            student_name <- "" #@param {type:"string"}

            submission_time <- Sys.time()

            if (!nzchar(trimws(student_name))) {
              stop("氏名を漢字で入力してから、このセルを再実行してください。")
            }

            cat("氏名:", student_name, "\n")
            cat("解析した細胞種:", celltype, "\n")
            cat("実行時刻:", format(submission_time, "%Y-%m-%d %H:%M:%S %Z"), "\n")
            